In [1]:
#S26#

In [2]:
###Lasso, ElasticNet & Sparsity###

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso

# ==========================================
# 1. Load dataset
# ==========================================
df = pd.read_csv(r'C:/Users/ranja/Downloads/cp_data_demo.csv')

# Remove spaces from column names
df.columns = df.columns.str.strip()

print("Columns in dataset:")
print(df.columns.tolist())


# ==========================================
# 2. Define target
# ==========================================
y = df['Temperature']


# ==========================================
# 3. Define features
# ==========================================

# Remove target column
X = df.drop(columns=['Temperature'], errors='ignore')

# Keep only numerical columns
X = X.select_dtypes(include=['number'])

print("\nFeatures used:")
print(X.columns.tolist())


# ==========================================
# 4. Handle missing values
# ==========================================
X = X.fillna(X.median())
y = y.fillna(y.median())


# ==========================================
# 5. Split data
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ==========================================
# 6. Train Lasso
# ==========================================
lasso = Lasso(
    alpha=1.0,
    random_state=42,
    selection='random'
)

lasso.fit(X_train, y_train)


# ==========================================
# 7. Predictions
# ==========================================
lasso_preds = lasso.predict(X_test)


# ==========================================
# 8. Find zeroed features
# ==========================================
zeroed_features = X_train.columns[lasso.coef_ == 0]

print("\nNumber of features:", len(X_train.columns))
print("Number of zeroed features:", len(zeroed_features))

print("\nZeroed features:")
print(zeroed_features.tolist())


Columns in dataset:
['Formula', 'Temperature', 'Heat_Capacity']

Features used:
['Heat_Capacity']

Number of features: 1
Number of zeroed features: 0

Zeroed features:
[]


In [7]:
# 2. Train ElasticNet
elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
elastic.fit(X_train, y_train)
elastic_preds = elastic.predict(X_test)

In [8]:
# 3. Define Evaluation Helper
def evaluate_model(y_true, y_pred):
    return {
        'R²': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred)
    }

In [14]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV
import pandas as pd

# ---------------------------------
# 1. Linear Regression
# ---------------------------------
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_preds = lr.predict(X_test)


# ---------------------------------
# 2. Ridge Regression - Best Alpha
# ---------------------------------
ridge_grid = GridSearchCV(
    Ridge(),
    {'alpha': [0.01, 0.1, 1, 10, 100]},
    cv=5,
    scoring='neg_mean_squared_error'
)

ridge_grid.fit(X_train, y_train)

best_alpha = ridge_grid.best_params_['alpha']

ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train, y_train)

ridge_preds = ridge.predict(X_test)


# ---------------------------------
# 3. Lasso Regression
# ---------------------------------
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)

lasso_preds = lasso.predict(X_test)


# ---------------------------------
# 4. ElasticNet Regression
# ---------------------------------
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
elastic.fit(X_train, y_train)

elastic_preds = elastic.predict(X_test)


# ---------------------------------
# 5. Model Evaluation Function
# ---------------------------------
def evaluate_model(y_true, y_pred):
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R² Score': r2_score(y_true, y_pred)
    }


# ---------------------------------
# 6. Create Comparison Table
# ---------------------------------
results = pd.DataFrame({
    'Unregularized LR': evaluate_model(y_test, lr_preds),
    'Ridge (L2)': evaluate_model(y_test, ridge_preds),
    'Lasso (L1)': evaluate_model(y_test, lasso_preds),
    'ElasticNet': evaluate_model(y_test, elastic_preds)
})

print("Best Ridge Alpha:", best_alpha)

results

Best Ridge Alpha: 100


,Unregularized LR,Ridge (L2),Lasso (L1),ElasticNet
MAE,573.601962,573.601966,573.601974,573.601975
MSE,535899.774863,535899.751710,535899.702161,535899.696132
RMSE,732.051757,732.051741,732.051707,732.051703
R² Score,0.003915,0.003915,0.003915,0.003915


In [15]:
# Save the deliverable
results.to_csv('regularization_comparison_s26.csv')
print("✅ 4-model comparison table successfully saved.")

✅ 4-model comparison table successfully saved.
